# Lab Task 2 - Linear Regression
**Name:** [Your Name]  
**Roll No:** [Your Roll No]

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

## Dataset 1 - Temperature vs Electricity Consumption

In [ ]:
df1 = pd.read_csv(r'c:\Users\Geo Computer\OneDrive\Desktop\Farhan\temperature_electricity.csv')
df1.head()

In [ ]:
df1.describe()

In [ ]:
df1.isnull().sum()

In [ ]:
plt.scatter(df1['Temperature_C'], df1['Electricity_kWh'], color='blue')
plt.xlabel('Temperature (C)')
plt.ylabel('Electricity (kWh)')
plt.title('Temperature vs Electricity')
plt.show()

In [ ]:
X1 = df1[['Temperature_C']]
y1 = df1['Electricity_kWh']

X1_train, X1_test, y1_train, y1_test = train_test_split(X1, y1, test_size=0.2, random_state=42)

model1 = LinearRegression()
model1.fit(X1_train, y1_train)

y1_pred = model1.predict(X1_test)

print('Intercept:', model1.intercept_)
print('Coefficient:', model1.coef_)
print('MSE:', mean_squared_error(y1_test, y1_pred))
print('R2 Score:', r2_score(y1_test, y1_pred))

In [ ]:
plt.scatter(X1, y1, color='blue', label='Data')
plt.plot(X1, model1.predict(X1), color='red', label='Regression Line')
plt.xlabel('Temperature (C)')
plt.ylabel('Electricity (kWh)')
plt.title('Linear Regression - Dataset 1')
plt.legend()
plt.show()

## Dataset 2 - Car Age vs Car Price (PKR)

In [ ]:
df2 = pd.read_csv(r'c:\Users\Geo Computer\OneDrive\Desktop\Farhan\car_age_price.csv')
df2.head()

In [ ]:
df2.describe()

In [ ]:
df2.isnull().sum()

In [ ]:
plt.scatter(df2['Car_Age_Years'], df2['Price_PKR_Thousands'], color='orange')
plt.xlabel('Car Age (Years)')
plt.ylabel('Price (PKR Thousands)')
plt.title('Car Age vs Price')
plt.show()

In [ ]:
X2 = df2[['Car_Age_Years']]
y2 = df2['Price_PKR_Thousands']

X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.2, random_state=42)

model2 = LinearRegression()
model2.fit(X2_train, y2_train)

y2_pred = model2.predict(X2_test)

print('Intercept:', model2.intercept_)
print('Coefficient:', model2.coef_)
print('MSE:', mean_squared_error(y2_test, y2_pred))
print('R2 Score:', r2_score(y2_test, y2_pred))

In [ ]:
plt.scatter(X2, y2, color='orange', label='Data')
plt.plot(X2, model2.predict(X2), color='green', label='Regression Line')
plt.xlabel('Car Age (Years)')
plt.ylabel('Price (PKR Thousands)')
plt.title('Linear Regression - Dataset 2')
plt.legend()
plt.show()

## Gradio Interface

In [ ]:
import gradio as gr

def predict_electricity(temp):
    result = model1.predict([[temp]])[0]
    return f'Electricity: {result:.2f} kWh'

def predict_price(age):
    result = model2.predict([[age]])[0]
    return f'Car Price: {result:.0f} PKR (Thousands)'

with gr.Blocks() as app:
    gr.Markdown('# Linear Regression Prediction')
    with gr.Tab('Dataset 1 - Electricity'):
        temp = gr.Number(label='Enter Temperature (C)')
        out1 = gr.Textbox(label='Prediction')
        gr.Button('Predict').click(predict_electricity, temp, out1)
    with gr.Tab('Dataset 2 - Car Price'):
        age = gr.Number(label='Enter Car Age (Years)')
        out2 = gr.Textbox(label='Prediction')
        gr.Button('Predict').click(predict_price, age, out2)

app.launch()

## Flask Interface
Run this cell. It will print a **clickable link**. Click it to open the app in your browser.

In [ ]:
from flask import Flask, request, jsonify, render_template_string
import threading
from IPython.display import display, HTML

app2 = Flask(__name__)

html_page = """
<!DOCTYPE html>
<html>
<head>
    <title>Linear Regression App</title>
    <style>
        body { font-family: Arial; background: #f0f0f0; padding: 30px; }
        .box { background: white; padding: 20px; margin: 15px 0; border-radius: 10px; width: 380px; }
        h3 { color: #333; }
        input { padding: 8px; width: 220px; margin: 6px 0; font-size: 14px; }
        button { margin-top: 8px; padding: 9px 18px; background: #007bff; color: white; border: none; border-radius: 5px; cursor: pointer; font-size: 14px; }
        .result { margin-top: 10px; color: green; font-weight: bold; font-size: 15px; }
    </style>
</head>
<body>
    <h1>Linear Regression - Lab Task 2</h1>

    <div class="box">
        <h3>Dataset 1 - Predict Electricity (kWh)</h3>
        <input type="number" id="temp" placeholder="Enter Temperature in C" /><br>
        <button onclick="getElec()">Predict</button>
        <div class="result" id="r1"></div>
    </div>

    <div class="box">
        <h3>Dataset 2 - Predict Car Price (PKR)</h3>
        <input type="number" id="age" placeholder="Enter Car Age in Years" /><br>
        <button onclick="getPrice()">Predict</button>
        <div class="result" id="r2"></div>
    </div>

    <script>
        function getElec() {
            var t = document.getElementById('temp').value;
            fetch('/p1?temp=' + t)
                .then(res => res.json())
                .then(data => { document.getElementById('r1').innerText = data.result; });
        }
        function getPrice() {
            var a = document.getElementById('age').value;
            fetch('/p2?age=' + a)
                .then(res => res.json())
                .then(data => { document.getElementById('r2').innerText = data.result; });
        }
    </script>
</body>
</html>
"""

@app2.route('/')
def home():
    return render_template_string(html_page)

@app2.route('/p1')
def p1():
    temp = float(request.args.get('temp'))
    pred = model1.predict([[temp]])[0]
    return jsonify({'result': f'Predicted Electricity: {pred:.2f} kWh'})

@app2.route('/p2')
def p2():
    age = float(request.args.get('age'))
    pred = model2.predict([[age]])[0]
    return jsonify({'result': f'Predicted Car Price: {pred:.0f} PKR (Thousands)'})

thread = threading.Thread(target=lambda: app2.run(port=5000, debug=False, use_reloader=False))
thread.daemon = True
thread.start()

print('Server is running...')
display(HTML('<br><a href="http://127.0.0.1:5000" target="_blank" style="font-size:16px; color:blue; font-weight:bold;">>>> Click here to open Flask App <<<</a>'))